# BAL Cell type counts neutrophil normalization / analysis

In this notebook we will analyze BAL data cell type counts, and normalized them with respect to flow data neutrophil percentage.

We will:
- group BAL samples w.r.t. `BAL_BARCODE_NAME` w.r.t. `CELLTYPE_LEVEL_COLNAME` counts,
- normalize them without neutrophil information from the flow data and store information in `BAL_NORMED_CELL_TYPE_COLNAME_PREFIX` prefixed cell type columns,
- normalize them with neutrophil information from the flow data and store information `BAL_NORMED_CELL_TYPE_COLNAME_PREFIX` prefixed cell type columns,
- compute expected neutrophil count and store it in `EXPECTED_NEUTROPHIL_COUNT_COLNAME`.

In [ ]:
import sys
sys.path.insert(0, '../lib')

In [ ]:
import pandas
import scanpy
import seaborn

import common_data

In [ ]:
# Data paths:
BAL_DATA_PATH = common_data.SC_NORM
FLOW_PATH = common_data.RAW_FLOW
RESULT_DATA_PATH = common_data.DATA / '2023-10-13-nn_lympho_flow.csv'

# Columns from existing DFs names:
BAL_BARCODE_COLNAME = 'bal_barcode'
CELLTYPE_LEVEL_COLNAME = 'Level_6'
FLOW_DATA_NEUTROPHIL_COLNAME = 'Neutrophils percent'
GROUPING_COLNAME = 'individual'

# New column names:
BAL_NEUTROPHIL_NORMED_CELL_TYPE_COLNAME_PREFIX = 'LYMPHO_NNORMED'
BAL_NORMED_CELL_TYPE_COLNAME_PREFIX = 'NORMED'
EXPECTED_NEUTROPHIL_COUNT_COLNAME = 'ExpNeutrophil'
FLOW_COLNAME_PREFIX = 'FLOW'
FLOW_PREFIXED_NEUTROPHIL_COLNAME = f'{FLOW_COLNAME_PREFIX}:{FLOW_DATA_NEUTROPHIL_COLNAME}'
NN_PREFIXED_NEUTROPHIL_COLNAME = f'{BAL_NEUTROPHIL_NORMED_CELL_TYPE_COLNAME_PREFIX}:{FLOW_DATA_NEUTROPHIL_COLNAME}'

In [2]:
flow_df = pandas.read_csv(FLOW_PATH)
bal_data = scanpy.read_h5ad(BAL_DATA_PATH)

Grouping BAL data w.r.t. to `GROUPING_COLNAME` and counting cell types from `CELLTYPE_LEVEL_COLNAME`:

In [3]:
# Grouping BAL data through patients and celltypes
cell_type_count_data = pandas.DataFrame(bal_data.obs.groupby([GROUPING_COLNAME, CELLTYPE_LEVEL_COLNAME]).size()).unstack(fill_value=0)

# Renaming the columns to cell types
cell_type_count_data.columns = [ccd[1] for ccd in cell_type_count_data.columns]

In [ ]:
lymph_columns = [
    'B cells',
    'CD4 T cells',
    'CD8 T cells',
    'Classical monocytes-1 CCR2',
    'Classical monocytes-2 IL1B',
    'DC1',
    'DC2',
    'Hematopoietic stem cells',
    'MRC1+C1QA+',
    'MRC1+C1QA-',
    'Migratory DC',
    'NUPR1+ Macs',
    'Non-classical monocytes',
    'Perivascular macrophages',
    'Plasma cells',
    'Proliferating CD4 T cells',
    'Proliferating CD8 T cells',
    'Proliferating NUPR1+ Macs',
    'Proliferating gdT cells',
    'Proliferating plasma cells',
    'Tregs',
    'gdT cells',
    'pDC'
]

In [5]:
cell_type_count_data = cell_type_count_data[lymph_columns]

In [6]:
print(f'Number of patients in BALs = {len(set(cell_type_count_data.index))}.')
print(f'Number of patients in FLOW data = {len(set(flow_df[BAL_BARCODE_COLNAME]))}.')
print(f'Intersection of above = {len(set(flow_df[BAL_BARCODE_COLNAME]) & set(cell_type_count_data.index))}.')

Number of patients in BALs = 303.
Number of patients in FLOW data = 792.
Intersection of above = 241.


Now we will join created dataframe with flow dataframe:

In [7]:
bal_flow_sample_intersection = list(set(flow_df[BAL_BARCODE_COLNAME]) & set(cell_type_count_data.index))

In [8]:
bb_indexed_flow_df = flow_df.set_index(BAL_BARCODE_COLNAME)

In [9]:
flow_cell_type_columns = flow_df.columns[4:]
print(flow_cell_type_columns)

Index(['CD4 T cells percent', 'CD8 T cells percent', 'Treg cells percent',
       'Neutrophils percent', 'CD206high macs percent',
       'CD206low macs percent', 'Monocytes percent'],
      dtype='object')


In [10]:
prefixed_flow_df_columns = [col if col not in flow_cell_type_columns else f'{FLOW_COLNAME_PREFIX}:{col}' for col in bb_indexed_flow_df.columns]
print(prefixed_flow_df_columns)

['Unnamed: 0', 'sample_id', 'panel', 'FLOW:CD4 T cells percent', 'FLOW:CD8 T cells percent', 'FLOW:Treg cells percent', 'FLOW:Neutrophils percent', 'FLOW:CD206high macs percent', 'FLOW:CD206low macs percent', 'FLOW:Monocytes percent']


In [11]:
bb_indexed_flow_df.columns = prefixed_flow_df_columns

In [12]:
flow_prefiexed_cell_type_names = prefixed_flow_df_columns[3:]

In [13]:
flow_prefiexed_cell_type_names

['FLOW:CD4 T cells percent',
 'FLOW:CD8 T cells percent',
 'FLOW:Treg cells percent',
 'FLOW:Neutrophils percent',
 'FLOW:CD206high macs percent',
 'FLOW:CD206low macs percent',
 'FLOW:Monocytes percent']

In [14]:
flow_joined_cell_type_count_data = cell_type_count_data.loc[bal_flow_sample_intersection]
flow_joined_cell_type_count_data = flow_joined_cell_type_count_data.join(bb_indexed_flow_df)

### Neutrophil and sample normalization

First let's prepare appropriate statistics and columns names:

In [15]:
bal_cell_type_columns = cell_type_count_data.columns

In [16]:
bal_cell_type_count_data_sums = flow_joined_cell_type_count_data[bal_cell_type_columns].sum(axis=1)

In [ ]:
normed_prefixed_cell_type_columns = [
    f':{BAL_NORMED_CELL_TYPE_COLNAME_PREFIX}:{col}'
    for col in bal_cell_type_columns
]

In [ ]:
bal_neutrophil_prefixed_cell_type_columns = [
    f'{BAL_NEUTROPHIL_NORMED_CELL_TYPE_COLNAME_PREFIX}:{col}'
    for col in bal_cell_type_columns
]

In [19]:
all_neutrophil_prefixed_cell_type_columns = bal_neutrophil_prefixed_cell_type_columns + [NN_PREFIXED_NEUTROPHIL_COLNAME]

First - let us normalize with respect to sample:

In [20]:
flow_joined_cell_type_count_data[
    normed_prefixed_cell_type_columns
] = flow_joined_cell_type_count_data[bal_cell_type_columns].to_numpy() / bal_cell_type_count_data_sums.to_numpy().reshape((-1, 1))

Now - let us compute the neutrophil normalized data. We will do this by multiplying each percentage by (1.0 - `neutrophil_percentage`), so that non-neutrophil percentages sums up to 1.0 - `neutrophil_percentage`.

In [21]:
flow_joined_cell_type_count_data[
    bal_neutrophil_prefixed_cell_type_columns
] = flow_joined_cell_type_count_data[
    normed_prefixed_cell_type_columns
].to_numpy() * (1.0 - flow_joined_cell_type_count_data[FLOW_PREFIXED_NEUTROPHIL_COLNAME].to_numpy().reshape((-1, 1)) / 100.0)

In [ ]:
flow_joined_cell_type_count_data[NN_PREFIXED_NEUTROPHIL_COLNAME] = flow_joined_cell_type_count_data[FLOW_PREFIXED_NEUTROPHIL_COLNAME] / 100.0

Test: if the process was successful the overall counts should sum up to `1.0`.

In [ ]:
flow_joined_cell_type_count_data[
    bal_neutrophil_prefixed_cell_type_columns
].sum(axis=1) + flow_joined_cell_type_count_data[FLOW_PREFIXED_NEUTROPHIL_COLNAME] / 100

Now - let us compute the expected neutrophil counts. Let `count` be a sum of all non-neutrophil cell type counts. To compute neutrophil normalized counts we need to:
- first compute the overall count which is equal to `total_count = count / (1 - neutrophil_percentage)`,
- now the expected neutrophil count is equal to `total_count * neutrophil_percentage`,
- this is equivalent to dividing `count` by `(1 - neutrophil_percentage) / neutrophil_percentage`).

The result will be stored in `EXPECTED_NEUTROPHIL_COUNT_COLNAME`.

In [24]:
neutrophil_flow_percentage = flow_joined_cell_type_count_data[FLOW_PREFIXED_NEUTROPHIL_COLNAME].to_numpy() / 100.0
neutrophil_normalization_factor = (1.0 - neutrophil_flow_percentage) / neutrophil_flow_percentage

flow_joined_cell_type_count_data[
    EXPECTED_NEUTROPHIL_COUNT_COLNAME
] = bal_cell_type_count_data_sums.to_numpy() / neutrophil_normalization_factor

In [25]:
all_count_columns = list(bal_cell_type_columns) + [EXPECTED_NEUTROPHIL_COUNT_COLNAME]

Test: Neutrophil normalized-percentages should agree with normalized cell type counts that include expected neutrophil count. Let us check that:

In [26]:
exp_count_data = flow_joined_cell_type_count_data[all_count_columns]
assert (exp_count_data / exp_count_data.sum(axis=1).to_numpy().reshape((-1, 1)) == exp_count_data / exp_count_data.sum(axis=1).to_numpy().reshape((-1, 1))).to_numpy().mean() == 1.0

## Data saving:

In [27]:
flow_joined_cell_type_count_data.to_csv(RESULT_DATA_PATH)